# Kernel: corn for grain and corn for silage, by state and for the US

**What USDA asked for:** two numbers, acres of corn harvested for grain and acres harvested for silage, for
each state and the US, before NASS's January report. This notebook makes the 2026 numbers from data public
by mid-November 2026, then shows how the same method did against NASS's final acres in 2017–2025.

**Runs offline in about a minute.** Inputs are in `data/` here (NASS Quick Stats extracts) plus the
organizer file `../02_QuickStats/corn_grain_silage_acres_2012_2025.csv`. Code is the `kernelag` package
here; full project: the team repository (`kernelag/deliverable.py`, `kernelag/silage_level.py`,
`kernelag/state_model.py`).

## Method

**Silage** (NASS publishes no in-season figure, so this is our own estimate):

* *Smoothed state level.* Most of a state's yearly swing in silage acres reverses the next year (section 3
  measures it). So next year's normal acres are a smoothed level (exponential smoothing in logs), not last
  year's figure.
* *Drought chopping.* When NASS's November yield forecast falls more than 10% below the state's 10-year
  trend, farmers chop grain corn for silage: `chopped = γ × max(0, shortfall − 0.10) × grain acres`.
* Two parameters (smoothing weight α, chopping rate γ), chosen for each year **on earlier years only**.

**Grain:** NASS's own November harvested-for-grain forecast where it publishes one, so **grain accuracy is
NASS's, not ours**. We also score a grain number of our own that does not use that forecast: in-season
planted acres × last year's grain share.

**Intervals:** quantiles of the backtest's own errors, pooled within three state size classes.

In [1]:
import os, sys
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE))                                  # the kernelag package next to this notebook
os.environ.setdefault("KERNEL_DATA_DIR", str(HERE / "data"))
if (HERE.parent / "02_QuickStats").is_dir():                   # organizer kit = the workspace root
    os.environ.setdefault("KERNEL_KIT_DIR", str(HERE.parent))

import warnings
import numpy as np
import pandas as pd
from kernelag import deliverable

warnings.filterwarnings("ignore")
np.seterr(all="ignore")
pd.set_option("display.width", 200)
result = deliverable.run(lean=True)      # writes data/models/deliverable_*.csv and deliverable_summary.json
summary, comparison = result["summary"], result["comparison"]
print("truth check (organizer CSV vs our NASS copy):", summary["truth_check"])

truth check (organizer CSV vs our NASS copy): {'organizer_rows_also_in_pipeline_copy': 979, 'rows_that_differ': 0}


## 1. The 2026 answer

In [2]:
f = result["deliverable"]
us = f[f["state"].str.startswith("US")].iloc[0]
pd.DataFrame({
    "acres (millions)": [us["silage"] / 1e6, us["grain"] / 1e6],
    "80% low": [us["silage_lo80"] / 1e6, us["grain_lo80"] / 1e6],
    "80% high": [us["silage_hi80"] / 1e6, us["grain_hi80"] / 1e6],
    "90% low": [us["silage_lo90"] / 1e6, us["grain_lo90"] / 1e6],
    "90% high": [us["silage_hi90"] / 1e6, us["grain_hi90"] / 1e6],
}, index=["Corn for silage, US 2026", "Corn for grain, US 2026"]).round(2)

,acres (millions),80% low,80% high,90% low,90% high
"Corn for silage, US 2026",6.26,5.95,6.59,5.86,6.68
"Corn for grain, US 2026",88.51,87.27,89.76,86.92,90.12


In [3]:
states = f[~f["state"].str.startswith("US")][["state", "silage", "silage_lo80", "silage_hi80",
                                                 "grain", "grain_lo80", "grain_hi80", "grain_method"]]
states.set_index("state").sort_values("silage", ascending=False).head(20)

,silage,silage_lo80,silage_hi80,grain,grain_lo80,grain_hi80,grain_method
state,,,,,,,
WISCONSIN,830100.0,707600.0,1060500.0,3100000.0,2998800.0,3151500.0,nass_forecast
NEW YORK,462800.0,394500.0,591300.0,535000.0,492600.0,549200.0,nass_forecast
SOUTH DAKOTA,424100.0,361500.0,541800.0,5840000.0,5649300.0,5937000.0,nass_forecast
MINNESOTA,378400.0,322500.0,483400.0,8300000.0,8028900.0,8437800.0,nass_forecast
CALIFORNIA,365500.0,311600.0,467000.0,50000.0,39700.0,58700.0,nass_forecast
MICHIGAN,332500.0,283400.0,424800.0,1950000.0,1886300.0,1982400.0,nass_forecast
PENNSYLVANIA,326900.0,278600.0,417500.0,710000.0,653700.0,728800.0,nass_forecast
IOWA,305200.0,260100.0,389900.0,12800000.0,12382000.0,13012600.0,nass_forecast
KANSAS,275400.0,234700.0,351800.0,6550000.0,6336100.0,6658800.0,nass_forecast


## 2. How the method did, 2017–2024 and 2025

Truth is NASS's final survey acres from the organizer CSV. That CSV has no state survey rows for the
Census years 2017 and 2022; those two years come from our NASS Quick Stats copy (`truth_source`).

* **State error** = Σ |forecast − actual| over all state-years ÷ Σ actual acres (acreage-weighted, pooled).
* **US error** = Σ over years |US forecast − US actual| ÷ Σ US actual (each year's US total, pooled).

2017–2024 also chose the method, so it is validation, not a clean test. 2025 was scored once, after the
method was fixed.

In [4]:
def pooled(series, years, column="predicted"):
    rows = comparison[(comparison["series"] == series) & comparison["year"].isin(years)].dropna(subset=["last"])
    st, us = rows[~rows["state"].str.startswith("US")], rows[rows["state"].str.startswith("US")]
    err = lambda p, a: 100 * (p - a).abs().sum() / a.sum()
    return {"state-years": len(st), "state error %": err(st[column], st["actual"]),
            "US years": len(us), "US error %": err(us[column], us["actual"])}

table = {}
for label, years in [("2017–2024", range(2017, 2025)), ("2025 held out", [2025])]:
    table[("Silage, Kernel", label)] = pooled("silage", years)
    table[("Silage, last year's acres", label)] = pooled("silage", years, "last")
    table[("Grain, delivered (= NASS Nov forecast)", label)] = pooled("grain", years)
    table[("Grain, ours without NASS forecast", label)] = pooled("grain_without_nass_forecast", years)
    table[("Grain, last year's acres", label)] = pooled("grain", years, "last")
pd.DataFrame(table).T.round(2)

,,state-years,state error %,US years,US error %
"Silage, Kernel",2017–2024,384.0,12.79,8.0,3.36
"Silage, last year's acres",2017–2024,384.0,14.50,8.0,4.09
"Grain, delivered (= NASS Nov forecast)",2017–2024,328.0,1.64,8.0,0.80
"Grain, ours without NASS forecast",2017–2024,328.0,2.34,8.0,1.04
"Grain, last year's acres",2017–2024,328.0,7.36,8.0,4.13
"Silage, Kernel",2025 held out,48.0,13.12,1.0,2.19
"Silage, last year's acres",2025 held out,48.0,13.96,1.0,0.32
"Grain, delivered (= NASS Nov forecast)",2025 held out,41.0,1.64,1.0,0.99
"Grain, ours without NASS forecast",2025 held out,41.0,1.63,1.0,0.69
"Grain, last year's acres",2025 held out,41.0,9.23,1.0,8.69


In [5]:
by_year = pd.DataFrame({
    (s, k): {y: v[k] for y, v in summary[s]["by_year"].items()}
    for s in ["silage", "grain", "grain_without_nass_forecast"]
    for k in ["state_wape_pct", "us_error_pct", "last_year_state_wape_pct", "last_year_us_error_pct"]})
by_year.index.name = "year"
by_year["silage"]

,state_wape_pct,us_error_pct,last_year_state_wape_pct,last_year_us_error_pct
year,,,,
2017,11.55,-2.91,12.33,-2.80
2018,17.74,3.04,20.25,4.84
2019,14.65,-6.13,17.20,-7.24
2020,9.78,-3.85,12.03,0.08
2021,11.36,-0.20,14.43,1.69
2022,15.66,-5.52,14.16,-5.84
2023,9.51,0.41,13.53,6.04
2024,12.13,4.63,12.34,4.09
2025,13.12,2.19,13.96,0.32


## 3. What limits accuracy

Measured in the repository's `test_models/accuracy/noise_floor.py` (NASS state survey 2000–2024; 2025 not
read) and copied here as `data/noise_floor_results.json`; the rest is computed from the comparison above.

In [6]:
import json
nf = json.loads((HERE / "data" / "noise_floor_results.json").read_text())
d, cg = nf["decomposition"], nf["census_vs_survey"]
rows = comparison[(comparison["series"] == "silage") & comparison["year"].between(2017, 2024)
                  & ~comparison["state"].str.startswith("US")].copy()
size = rows.groupby("state")["last"].mean()
third = pd.qcut(size.rank(method="first"), 3, labels=["smallest", "middle", "largest"])
rows["third"] = rows["state"].map(third)
points = (rows["predicted"] - rows["actual"]).abs().groupby(rows["third"], observed=True).sum() / rows["actual"].sum() * 100
g = pooled("grain", range(2017, 2025))["state error %"]
own = pooled("grain_without_nass_forecast", range(2017, 2025))["state error %"]
print(f"Lag-1 autocorrelation of a state's own silage change: {d['state_change_lag1_autocorr']:.2f} (pure noise -0.5)")
print(f"Implied state error from that noise alone: about {d['noise_floor_state_wape_pct']:.1f}%")
print(f"Share of yearly change common to all states: {d['national_share_of_change_variance']:.2f}")
print(f"NASS survey vs Census, state gap: 2017 {cg['2017']['state_gap_pct']}%, 2022 {cg['2022']['state_gap_pct']}%")
print("Silage error points by state size third, 2017-2024:", points.round(2).to_dict())
print(f"Grain: our own number is {own - g:.2f} points of state error behind NASS's forecast")

Lag-1 autocorrelation of a state's own silage change: -0.40 (pure noise -0.5)
Implied state error from that noise alone: about 10.1%
Share of yearly change common to all states: 0.11
NASS survey vs Census, state gap: 2017 15.6%, 2022 32.1%
Silage error points by state size third, 2017-2024: {'smallest': 0.44, 'middle': 2.58, 'largest': 9.76}
Grain: our own number is 0.70 points of state error behind NASS's forecast


* **Silage state error is mostly noise in NASS's own state series**, and it sits in the big dairy
  states. Every model the team tried (ridge, LAD, Huber, gradient boosting, ensembles) lands near the same
  level; only new data on those states (e.g. a calibrated satellite signal) could move it.
* **The US silage total is where the model helps:** drought chopping turns a national weather shock into acres.
* **Grain** is NASS's November forecast; our own number is close behind it.
* **Satellite classifier** (separate work in the repository): it separates silage from grain fields on the
  Maryland research farm, but its year-to-year drift is too large to move state acreage yet, so it is not
  used in these numbers.
* All replays use current-vintage NASS data, not the figures as published each November.